# Import Libraries and Access BigQuery

I queried everything from the fact table I made earlier and put it into a dataframe.

In [62]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

auth.authenticate_user()
project_id = 'healthcare-analysis-489903'
client = bigquery.Client(project=project_id)

In [63]:
query = f"SELECT * FROM `{project_id}.stg_nhanes.fct_cardiovascular_risk`"
df = client.query(query).to_dataframe()

# Preprocessing

To predict a patient with high cholesterol, I used a variety of variables that may impact cholesterol. Then I encoded variables so the model could read them and defined them as high risk.


In [64]:
df['is_high_risk'] = (df['cholesterol_status'] == 'High').astype(int)

features = [
    'age', 'bmi', 'avg_systolic_bp', 'avg_diastolic_bp',
    'daily_calories', 'sodium_mg',
    'smoker_status', 'physical_activity'
]

X_all = pd.get_dummies(df[features], columns=['smoker_status', 'physical_activity'], drop_first=True)
y_all = df['is_high_risk']

# Train/Test Split
I split the model into the normal train/test, but decided to implement stratification to ensure the model can train on higher risk patients in defense of a "bad" seed. Next, I used the median on the training set and applied it directly onto the test set to avoid leakage

In [65]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
median_vals = X_train.median()
X_train = X_train.fillna(median_vals)
X_test = X_test.fillna(median_vals)
X_all_filled = X_all.fillna(median_vals)

# Model Creation and Class Balancing
Used scale_pos_weight to prioritize high risk patients and prioritize them recall. I also kept the learning rate slow and prevented overfitting.
XG Boost was chosen since healthy patients outnumbered high risk ones and there were so many missing values that I decided that instead of imputing them all letting XG boost do the heavy lifting sufficed.

In [66]:
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

# Recall Focus and Model Results

I lowered the threshold to 0.25 to make my model more sensitive. This lowered the prevision but it incrased recall to 0.83, making it more likely that a high risk patient wouldn't be missed. The ROC AUC score being above 0.7 means my model is pretty good at determining a high risk patient from a normal one.

In [67]:
threshold = 0.25
y_probs_test = model.predict_proba(X_test)[:, 1]
y_pred_test = (y_probs_test >= threshold).astype(int)

print(classification_report(y_test, y_pred_test))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_probs_test):.2f}")

              precision    recall  f1-score   support

           0       0.98      0.59      0.74      1913
           1       0.12      0.83      0.20       122

    accuracy                           0.61      2035
   macro avg       0.55      0.71      0.47      2035
weighted avg       0.93      0.61      0.71      2035

ROC AUC Score: 0.74


# Deployment and Moving to PowerBI
Scored the risk probabilities and set a risk probability of >= 0.25 as 1 and < 0.25 as 0. This lets me put the model into PowerBI so it can be visualized.


In [68]:
df['risk_probability'] = model.predict_proba(X_all_filled)[:, 1]
df['model_prediction'] = (df['risk_probability'] >= 0.25).astype(int)
df.to_gbq("stg_nhanes.ml_cardiovascular_predictions", project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 8924.05it/s]
